In [1]:
!pip install flash-linear-attention

## FLA Simple Gated Linear Attention

In [ ]:
from fla.layers.simple_gla import SimpleGatedLinearAttention as sGLA

In [11]:
import torch
import time


BATCH_SIZE = 2
SEQ_LEN = 4096
HIDDEN_SIZE = 128
NUM_HEADS = 4
DTYPE = torch.float16
DEVICE = 'cuda'

model = sGLA(
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    mode='chunk', 
    fuse_norm=True,
    use_short_conv=False
)

model = model.to(device=DEVICE, dtype=DTYPE)
model.eval()

x = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_SIZE, device=DEVICE, dtype=DTYPE)

print(f"Benchmarking SimpleGatedLinearAttention ({model.mode} mode)...")
print(f"Input Shape: {x.shape}")

print("Warming up GPU...")
for _ in range(10):
    with torch.no_grad():
        _ = model(x)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()

n_loops = 100
for _ in range(n_loops):
    with torch.no_grad():
        y, _, _ = model(x)

end_event.record()
torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
avg_time = elapsed_time_ms / n_loops

print(f"Output shape: {y.shape}")
print(f"Total time for {n_loops} runs: {elapsed_time_ms:.2f} ms")
print(f"Average time per forward pass: {avg_time:.4f} ms")

Benchmarking SimpleGatedLinearAttention (chunk mode)...
Input Shape: torch.Size([2, 4096, 128])
Warming up GPU...
Output shape: torch.Size([2, 4096, 128])
Total time for 100 runs: 101.18 ms
Average time per forward pass: 1.0118 ms


### FLA Gated Linear Attention

In [12]:
from fla.layers.gla import GatedLinearAttention as GLA

In [20]:
import time
import torch

BATCH_SIZE = 2
SEQ_LEN = 4096
DTYPE = torch.float16
DEVICE = "cuda"

model = GLA(
    mode="chunk",
    hidden_size=512,
    expand_k=2.0,
    expand_v=2.0,
    num_heads=16,
    num_kv_heads=16,
    feature_map="relu",
    use_short_conv=True,
    conv_size=32,
    use_output_gate=True,
    gate_fn="gelu",
    fuse_norm=False,
    gate_low_rank_dim=128,
    clamp_min=-5.0,
)

model = model.to(device=DEVICE, dtype=DTYPE).eval()

x = torch.randn(BATCH_SIZE, SEQ_LEN, model.hidden_size, device=DEVICE, dtype=DTYPE)

attention_mask = torch.ones(BATCH_SIZE, SEQ_LEN, device=DEVICE, dtype=torch.int32)

print(f"Benchmarking GatedLinearAttention ({model.mode} mode)...")
print(f"Input Shape: {tuple(x.shape)} | dtype={x.dtype} | device={x.device}")

@torch.no_grad()
def run_once():
    y, _, _ = model(x, attention_mask=attention_mask)
    return y

print("Warming up GPU...")
for _ in range(10):
    _ = run_once()
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

n_loops = 100
start_event.record()
for _ in range(n_loops):
    y = run_once()
end_event.record()

torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
avg_time_ms = elapsed_time_ms / n_loops

print(f"Output shape: {tuple(y.shape)}")
print(f"Total time for {n_loops} runs: {elapsed_time_ms:.2f} ms")
print(f"Average time per forward pass: {avg_time_ms:.4f} ms")

Benchmarking GatedLinearAttention (chunk mode)...
Input Shape: (2, 4096, 512) | dtype=torch.float16 | device=cuda:0
Warming up GPU...
Output shape: (2, 4096, 512)
Total time for 100 runs: 515.68 ms
Average time per forward pass: 5.1568 ms
